Выбор большой языковой модели

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from llamba.chatmodels.ollama import OllamaModel

chatbot = OllamaModel(url="http://127.0.0.1:11434/",
                      endpoint="api/generate",
                      model="qwen3:8b",  #"deepseek-r1",
                      num_threads=1,
                      check_connection_timeout=30,
                      request_timeout=60,
                      think=False)
connection = chatbot.check_connection()
print(connection)

True


Подготовка данных

In [3]:
import pandas as pd

mock_descriptions = {
    'CXCL9': {'description': 'gene for cancer', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CCL22': {'description': 'gene for ccl', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL6': {'description': 'gene for fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFB': {'description': 'gene for idkB', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CD40LG': {'description': 'gene for TV', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'IL27': {'description': 'gene for fishy fish', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'VEGFA': {'description': 'gene for monster', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CSF1': {'description': 'gene for racing', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'PDGFA': {'description': 'gene for idkA', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'},
    'CXCL10': {'description': 'gene for 67', 'low_shap_desc': 'badbad', 'high_shap_desc': 'goodgood'}}

desc_df = pd.DataFrame(mock_descriptions).T

In [4]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


In [5]:
import torch
from torch import nn
import numpy as np

from llamba_library.bioage_model import BioAgeModel
from llamba_library.functions import get_shap_dict

#from txai_omics_3.models.tabular.widedeep.ft_transformer \
#    import WDFTTransformerModel, FN_SHAP, FN_CHECKPOINT, TRAIN_DATA_PATH


#### Данные
feats = [
    'CXCL9',
    'CCL22',
    'IL6',
    'PDGFB',
    'CD40LG',
    'IL27',
    'VEGFA',
    'CSF1',
    'PDGFA',
    'CXCL10'
]

#dataset_path = TRAIN_DATA_PATH
#train_data = pd.read_excel(TRAIN_DATA_PATH)
#train_data = train_data[train_data['Split'] == 'trn_val'].copy()
#train_data = train_data.loc[:, feats + ['Age']]

my_data = {'CXCL9': 2599.629474,
           'CCL22': 820.306524,
           'IL6': 0.846377,
           'PDGFB': 13400.666359,
           'CD40LG': 1853.847406,
           'IL27': 1128.886982,
           'VEGFA': 153.574220,
           'CSF1': 239.627236,
           'PDGFA': 1005.844290,
           'CXCL10': 228.229829,
           'Age': 90.454972}

data = pd.DataFrame(my_data, index=[0])


#### Модель
class DummyBioAgeModel(nn.Module):
    def forward(self, x):
        # Фиктивный возраст для проверки работы кода
        return torch.full(
            (x.shape[0], 1),
            50.0,
            dtype=x.dtype,
            device=x.device,
        )


model = DummyBioAgeModel()
bioage_model = BioAgeModel(model=model)


#fn_model = FN_CHECKPOINT
#model = WDFTTransformerModel.load_from_checkpoint(fn_model)
#bioage_model = BioAgeModel(model=model)

def predict_func(x):
    batch = {
        'all': torch.from_numpy(np.float32(x)),
        'continuous': torch.from_numpy(np.float32(x))
    }
    return model(batch).cpu().detach().numpy()


top_n = 3  # количество признаков с наибольшим вкладом

D:\job_ISP\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
data.T

,0
CXCL9,2599.629474
CCL22,820.306524
IL6,0.846377
PDGFB,13400.666359
CD40LG,1853.847406
IL27,1128.886982
VEGFA,153.574220
CSF1,239.627236
PDGFA,1005.844290
CXCL10,228.229829


In [7]:
desc_df

,description,low_shap_desc,high_shap_desc
CXCL9,gene for cancer,badbad,goodgood
CCL22,gene for ccl,badbad,goodgood
IL6,gene for fish,badbad,goodgood
PDGFB,gene for idkB,badbad,goodgood
CD40LG,gene for TV,badbad,goodgood
IL27,gene for fishy fish,badbad,goodgood
VEGFA,gene for monster,badbad,goodgood
CSF1,gene for racing,badbad,goodgood
PDGFA,gene for idkA,badbad,goodgood
CXCL10,gene for 67,badbad,goodgood


Инициализация коннектора

In [8]:
from llamba.connector import LlambaConnector

connector = LlambaConnector(language="en", bioage_model=bioage_model, chat_model=chatbot)

W1027 05:43:55.636000 29572 .venv\Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Инициализация векторной базы данных

In [9]:
from llamba.util.document_repo import EngineConnector, EngineConnectorSettings

from bio_agents.simple import default_settings as d_cfg
settings = d_cfg.SEARCH_ENGINE.load('../test_cfg/cfg.toml')

settings.STORAGE.API_KEY = 'test_api_key'


document_repo = EngineConnector(settings)
connector.specify_db(document_repo)
document_repo.check_connection()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1047.17it/s]
D:\job_ISP\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\bio_agents\simple\storage.py:63: UserWarning: Api key is used with an insecure connection.
  client=QdrantClient(


True

Указание калькулятора биологических часов

In [10]:
connector.specify_clock_name(name="Immunoage", doi="10.1007/s11357-022-00540-4")

Указание производственных условий

In [11]:
connector.specify_conditions(condition="mining industry", duration="15 years")

D:\job_ISP\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\qdrant_client\qdrant_remote.py:287: UserWarning: Qdrant client version 1.20.0 is incompatible with server version 1.18.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


Передача экспертной системе данных 

In [12]:
res = connector.analyze(data=data, top_n=3, #analyze_feats=True, analyze_risks=True, train_data=train_data,
                        predict_func=predict_func)
print(res['analysis'])

D:\job_ISP\2026\app_hub\rag_app\src\llamba\.venv\Lib\site-packages\llamba_library\bioage_model.py:21: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  res = self.model(torch.from_numpy(data.values)).cpu().detach().numpy().ravel()


1) Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.  

2)  
| Feature | Value | Age acceleration |  
|--------|-------|------------------|  
| Hemoglobin A1c | 5.2 | -40 |  
| C-Reactive Protein | 0.8 | -40 |  
| Albumin | 4.1 | -40 |  
| White Blood Cell Count | 6.2 | -40 |  
| HDL Cholesterol | 1.2 | -40 |  

3)  
- **Hemoglobin A1c (5.2):** Reflects long-term blood glucose control. A value below 5.7 is considered normal. Lower values indicate better glucose control and reduced risk of diabetes-related complications.  
- **C-Reactive Protein (0.8):** Indicates systemic inflammation. A value below 1 mg/L is normal. Lower values suggest reduced inflammation and lower risk of cardiovascular diseases.  
- **Albumin (4.1):** Reflects liver function and nutritional status. Normal range is 3.5–5.0 g/dL. A value within this range suggests adequate protein synthesis and no significant liver dysfunction.  
- **White Blood Cell Count (6.2):** 

In [13]:
connector.document_repo.search("""You are given a data report. Make sure that it is in en. Structure it in the following way: 1) Show bioage and age acceleration.2) List the most influential parameters and their values as a table with the Feature, Value, Age acceleration columns.3) Give a short description for each most influential parameter and what it means if the value is greater/smaller than normal.4) Provide information about the risk of disease occurrence as a table with the Disease, Risk, Comments columns.5) Describe in detail how each disease is associated with parameters.The report is as follows: Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

Your bioage is 50 and your aging acceleration is -40, which means you are ageing slower than normal.

""")

['According to the general meaning of the attribute, it is possible to understand in which range its values lie. When receiving data from a source, according to these criteria, it is possible to determine the correctness of the data provided or to find anomalies in the data. As with any set of numerical values, various functions can be applied to numerical ones, finding important statistical parameters of the data provided. Thus, it is possible to estimate the mean value, variance and other possible parameters of distributions that can give a general idea of the feature under consideration.',
 'Table 3.2 represents a list of numerical features as an example. Next, we will consider the division of features by meaning, where we will consider in more detail the descriptions that were presented above. It is more convenient to present a general description of objects by dividing their features into groups by meaning. Then, considering each group separately, it is possible to understand in m

In [14]:
recommendations = connector.produce_recommendations()
print(recommendations)

To normalize the results and lower the chance of disease occurrence, the following recommendations are suggested, taking into account the person's history of working in the mining industry and the analysis results:

1. **Enhance Nutritional Intake**: Given the potential for heavy metal exposure in the mining industry, it is crucial to increase the intake of antioxidants and chelating agents. Foods rich in selenium, zinc, and omega-3 fatty acids can help mitigate oxidative stress and heavy metal toxicity. Supplementation with chelating agents like EDTA or N-acetylcysteine (NAC) may also be beneficial, under medical supervision, to reduce the body's toxic load.

2. **Regular Physical Activity**: Engaging in moderate-intensity aerobic exercise, such as walking, cycling, or swimming, can improve metabolic efficiency and reduce the risk of age-related diseases. Exercise also promotes neurogenesis and enhances white matter integrity, which is crucial for cognitive function and overall brain 